# 05 — Evaluation
Full evaluation of both models with metrics, plots, and comparison table.

In [1]:
import sys
sys.path.insert(0, '..')

import json
import pickle
import numpy as np
import torch

from src.config import (
    DEVICE, AE_LATENT_DIM, AE_HIDDEN_DIMS, CLS_HIDDEN_DIMS,
    AUTOENCODER_CHECKPOINT, CLASSIFIER_CHECKPOINT,
    EVAL_RESULTS_JSON, PROCESSED_DIR, BATCH_SIZE, NUM_CLASSES, CLASS_NAMES
)
from src.models import Autoencoder, Classifier
from src.train_utils import load_checkpoint
from src.dataset import make_loader
from src.eval_utils import (
    compute_reconstruction_errors, select_threshold, ae_predict,
    classifier_predict, binary_metrics, multiclass_metrics,
    get_confusion_matrix, get_roc_data
)
from src.visualize import (
    plot_reconstruction_error_dist, plot_confusion_matrix,
    plot_roc_curve, plot_tsne
)

print(f'Device: {DEVICE}')

Device: cuda


## Load data & models

In [2]:
with open(PROCESSED_DIR / 'data.pkl', 'rb') as f:
    data = pickle.load(f)

input_dim = data['X_test'].shape[1]

ae = Autoencoder(input_dim=input_dim, hidden_dims=AE_HIDDEN_DIMS, latent_dim=AE_LATENT_DIM)
ae = load_checkpoint(ae, AUTOENCODER_CHECKPOINT, DEVICE).to(DEVICE)

cls = Classifier(input_dim=input_dim, hidden_dims=CLS_HIDDEN_DIMS, num_classes=NUM_CLASSES)
cls = load_checkpoint(cls, CLASSIFIER_CHECKPOINT, DEVICE).to(DEVICE)

print('Models loaded.')

Models loaded.


## Autoencoder Evaluation

In [3]:
# Compute reconstruction errors
test_loader_ae = make_loader(data['X_test'], data['y_bin_test'], shuffle=False, batch_size=BATCH_SIZE)
train_normal_loader = make_loader(data['X_train'][data['normal_mask_train']], shuffle=False, batch_size=BATCH_SIZE)

train_normal_errors = compute_reconstruction_errors(ae, train_normal_loader, DEVICE)
test_errors = compute_reconstruction_errors(ae, test_loader_ae, DEVICE)

threshold = select_threshold(train_normal_errors)
print(f'Threshold (95th pct of normal train errors): {threshold:.6f}')

y_bin_pred_ae = ae_predict(test_errors, threshold)
ae_metrics = binary_metrics(data['y_bin_test'], y_bin_pred_ae)
print('\nAutoencoder binary metrics:')
for k, v in ae_metrics.items():
    print(f'  {k:<12} {v:.4f}')

Threshold (95th pct of normal train errors): 0.072365

Autoencoder binary metrics:
  accuracy     0.8712
  precision    0.9628
  recall       0.8049
  f1           0.8768


In [4]:
normal_mask_test = data['y_bin_test'] == 0
plot_reconstruction_error_dist(
    normal_errors=test_errors[normal_mask_test],
    attack_errors=test_errors[~normal_mask_test],
    threshold=threshold,
    filename='recon_error_dist.png'
)

Saved: /home/abzy/dev/aitu/masters/trimester3/aitu-multi-agent-systems/notebooks/../artifacts/recon_error_dist.png


PosixPath('/home/abzy/dev/aitu/masters/trimester3/aitu-multi-agent-systems/notebooks/../artifacts/recon_error_dist.png')

## Classifier Evaluation

In [5]:
test_loader_cls = make_loader(data['X_test'], data['y_cls_test'], shuffle=False, batch_size=BATCH_SIZE)
y_cls_pred, y_cls_prob = classifier_predict(cls, test_loader_cls, DEVICE)

cls_metrics = multiclass_metrics(data['y_cls_test'], y_cls_pred, y_cls_prob)
print('Classifier metrics:')
print(f"  Accuracy  : {cls_metrics['accuracy']:.4f}")
print(f"  Macro F1  : {cls_metrics['macro_f1']:.4f}")
print(f"  ROC-AUC   : {cls_metrics['roc_auc']:.4f}")

Classifier metrics:
  Accuracy  : 0.7674
  Macro F1  : 0.4886
  ROC-AUC   : 0.9252


In [6]:
cm = get_confusion_matrix(data['y_cls_test'], y_cls_pred)
plot_confusion_matrix(cm, CLASS_NAMES, filename='confusion_matrix.png')

Saved: /home/abzy/dev/aitu/masters/trimester3/aitu-multi-agent-systems/notebooks/../artifacts/confusion_matrix.png


PosixPath('/home/abzy/dev/aitu/masters/trimester3/aitu-multi-agent-systems/notebooks/../artifacts/confusion_matrix.png')

In [7]:
roc_data = get_roc_data(data['y_cls_test'], y_cls_prob)
plot_roc_curve(roc_data, filename='roc_curves.png')

Saved: /home/abzy/dev/aitu/masters/trimester3/aitu-multi-agent-systems/notebooks/../artifacts/roc_curves.png


PosixPath('/home/abzy/dev/aitu/masters/trimester3/aitu-multi-agent-systems/notebooks/../artifacts/roc_curves.png')

## t-SNE of Autoencoder Latent Space

In [8]:
ae.eval()
all_embeddings = []
with torch.no_grad():
    for batch in test_loader_ae:
        x = batch[0].to(DEVICE)
        z = ae.encode(x).cpu().numpy()
        all_embeddings.append(z)
embeddings = np.concatenate(all_embeddings)
plot_tsne(embeddings, data['y_cls_test'], CLASS_NAMES, filename='tsne_latent.png')

Running t-SNE (this may take a minute)...


Saved: /home/abzy/dev/aitu/masters/trimester3/aitu-multi-agent-systems/notebooks/../artifacts/tsne_latent.png


PosixPath('/home/abzy/dev/aitu/masters/trimester3/aitu-multi-agent-systems/notebooks/../artifacts/tsne_latent.png')

## Comparison Table

In [9]:
import pandas as pd

# Binary accuracy for classifier (normal vs. attack)
y_bin_pred_cls = (y_cls_pred != 0).astype(int)  # 0=normal, anything else=attack
from src.eval_utils import binary_metrics as bm
cls_binary = bm(data['y_bin_test'], y_bin_pred_cls)

comparison = pd.DataFrame({
    'Model': ['Autoencoder (unsupervised)', 'Classifier (supervised)'],
    'Accuracy': [ae_metrics['accuracy'], cls_binary['accuracy']],
    'Precision': [ae_metrics['precision'], cls_binary['precision']],
    'Recall': [ae_metrics['recall'], cls_binary['recall']],
    'F1': [ae_metrics['f1'], cls_binary['f1']],
})
comparison = comparison.set_index('Model')
comparison

,Accuracy,Precision,Recall,F1
Model,,,,
Autoencoder (unsupervised),0.871230,0.962808,0.804878,0.876788
Classifier (supervised),0.787482,0.973170,0.644432,0.775397


## Export results

In [10]:
eval_results = {
    'autoencoder': {
        'threshold': threshold,
        **ae_metrics,
    },
    'classifier': {
        'accuracy': cls_metrics['accuracy'],
        'macro_f1': cls_metrics['macro_f1'],
        'roc_auc': cls_metrics['roc_auc'],
        'binary': cls_binary,
        'per_class': cls_metrics['per_class'],
        'roc_data': {k: {kk: (vv if isinstance(vv, float) else vv) for kk, vv in v.items()} for k, v in roc_data.items()},
    },
}

with open(EVAL_RESULTS_JSON, 'w') as f:
    json.dump(eval_results, f, indent=2, default=str)
print(f'Saved: {EVAL_RESULTS_JSON}')

Saved: /home/abzy/dev/aitu/masters/trimester3/aitu-multi-agent-systems/notebooks/../artifacts/eval_results.json
